In [0]:
   # Contrôle : dlthub bien installé depuis le panneau (pas le module Databricks)
   import dlt
   print(dlt.__version__)   # attendu : 1.31.0
   print(dlt.__file__)      # attendu : chemin hors /databricks/

# API-Football : topredcards → landing (dlthub sur Databricks)

20 joueurs ayant reçu le plus de carton rouge  par compétition × saison (`/players/topscorers`), 1 appel par couple.
- `statistics` (toujours 1 élément) aplati en colonnes `statistics__…` : pas de table enfant.
- 6 colonnes toujours vides : type imposé (`TYPES_IMPOSES`).
- Incrémental : un couple déjà en landing ET terminé (d'après fixtures) n'est pas rappelé → lancer **après** le loader fixtures.

**Destination** : `/Volumes/<CATALOG>/landing/files/api_football/topredcards/<aaaa>/<mm>/<jj>/`

## 1. Paramètres (widgets)

`CATALOG` et `SECRET_SCOPE` sont toujours présents. Les autres widgets décrivent **ce qu'on demande à l'API** : À ADAPTER.

In [0]:
dbutils.widgets.text("CATALOG", "dev", "Catalog cible")
dbutils.widgets.text("SECRET_SCOPE", "scope-data-dev", "Secret scope")
# À ADAPTER : les widgets métier (exemple API-Football : championnats et saisons)
dbutils.widgets.text("LEAGUE_IDS", "61,39,78,140,135,2", "Ids championnats")
dbutils.widgets.text("SEASONS", "2022,2023,2024", "Saisons")

## 2. Installation de dlthub (version fixée)

## 3. Import de dlthub + contrôle « fail fast »

In [0]:
import dlt

if not hasattr(dlt, "__version__") or dlt.__file__.startswith("/databricks/"):
    raise ImportError("Conflit : le module 'dlt' chargé n'est pas dlthub (voir manuel 6, pièges)")
print("✅ dlthub", dlt.__version__)

## 4. Configuration

Toutes les valeurs propres à la source sont regroupées ici. **Aucune clé dans le code** : elle vient du secret scope.

In [0]:
import os, time
os.environ["RUNTIME__DLTHUB_TELEMETRY"] = "false"   # pas d'envoi de statistiques à dlthub
from dlt.sources.helpers import requests            # requests avec relances automatiques


def liste_entiers(texte: str) -> list[int]:
    """'61, 39,78' -> [61, 39, 78]"""
    return [int(x) for x in texte.split(",") if x.strip()]

CATALOG      = dbutils.widgets.get("CATALOG").strip()
SECRET_SCOPE = dbutils.widgets.get("SECRET_SCOPE").strip()

# ---------- À ADAPTER : identité de la source et de la table ----------

SOURCE   = "api_football"                         # nom du dossier ingestion/<source> (minuscules, _)
TABLE    = "topredcards"                             # nom de la table (minuscules, _)
BASE_URL = "https://v3.football.api-sports.io"    # adresse de l'API
ENDPOINT = "/players/topredcards"                            # chemin appelé
SECRET_KEY_NAME = "api-football-key"              # nom de la clé dans le scope ; None si l'API est sans clé
HEADER_NAME     = "x-apisports-key"               # en-tête qui porte la clé (voir la doc de l'API)
PAUSE_SECONDES  = 7                               # pause entre deux appels (limite par minute de l'API)
# ----------------------------------------------------------------------

# Colonnes vides sur tous les buteurs au 1er chargement (avertissement dlthub du 31/10) :
# on impose leur type pour qu'elles existent quand même (sinon dlthub ne les crée pas).
# Entier = même type que leurs colonnes sœurs remplies (dribbles__success, penalty__scored, goals__conceded…)
TYPES_IMPOSES = {
    "statistics__dribbles__past":    {"data_type": "bigint", "nullable": True},
    "statistics__games__number":     {"data_type": "bigint", "nullable": True},   # numéro de maillot
    "statistics__goals__saves":      {"data_type": "bigint", "nullable": True},
    "statistics__penalty__commited": {"data_type": "bigint", "nullable": True},   # orthographe de l'API (un seul t)
    "statistics__penalty__saved":    {"data_type": "bigint", "nullable": True},
    "statistics__penalty__won":      {"data_type": "bigint", "nullable": True},
}


HEADERS = {}
if SECRET_KEY_NAME:
    HEADERS[HEADER_NAME] = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY_NAME)

LANDING = f"/Volumes/{CATALOG}/landing/files"
print(f"Catalog={CATALOG} | Source={SOURCE} | Table={TABLE} | Landing={LANDING}/{SOURCE}/{TABLE}/")

## 5. Liste des appels à faire

Un **appel** = un dictionnaire de paramètres envoyé à l'API. À ADAPTER selon la source.

In [0]:
# À ADAPTER : construire la liste des appels à partir des widgets
LEAGUE_IDS = liste_entiers(dbutils.widgets.get("LEAGUE_IDS"))
SEASONS    = liste_entiers(dbutils.widgets.get("SEASONS"))
APPELS = [{"league": l, "season": s} for l in LEAGUE_IDS for s in SEASONS]

# Exemples pour d'autres sources (à mettre à la place) :
#   Open-Meteo : APPELS = [{"latitude": 47.25, "longitude": -1.52, "start_date": "2024-01-01",
#                           "end_date": "2024-12-31", "daily": "precipitation_sum,temperature_2m_mean"}]
#   Frankfurter : APPELS = [{"from": "EUR", "to": "GBP,USD,CHF"}]   (ENDPOINT = "/2022-01-01..2024-12-31")

print(f"{len(APPELS)} appel(s) prévu(s) : {APPELS[:5]}{' ...' if len(APPELS) > 5 else ''}")

## 5 bis. OPTIONNEL — Incrémental : ne pas rappeler ce qui est déjà complet en landing

Exemple API-Football : un couple championnat × saison est complet si tous ses matchs ont un statut final. À adapter (colonnes, règle) ou à **supprimer**.

In [0]:
# OPTIONNEL adapté à topscorers
# Règle : un couple compétition × saison n'est PAS rappelé s'il est :
#   1) déjà présent en landing topscorers, ET
#   2) terminé = tous ses matchs ont un statut final dans le DERNIER dépôt fixtures du couple.
# Une saison en cours est donc rappelée à chaque lancement (classement à jour).
# Prérequis : le loader fixtures tourne AVANT celui-ci (ordre à respecter dans l'orchestration).
# On ESSAIE de lire plutôt que tester os.path.isdir : le cache du montage /Volumes peut mentir
# (dossier supprimé ailleurs mais encore vu comme présent) -> Spark répond PATH_NOT_FOUND.
from pyspark.sql import functions as F, Window

STATUTS_FINAUX  = ["FT", "AET", "PEN", "AWD", "WO", "CANC", "ABD"]   # statuts de match définitifs (API-Football)
CHEMIN_TABLE    = f"{LANDING}/{SOURCE}/{TABLE}"                      # landing topscorers
CHEMIN_FIXTURES = f"{LANDING}/{SOURCE}/fixtures"                     # landing fixtures (sert à savoir si la saison est finie)

def absent(erreur: Exception) -> bool:
    """True si l'erreur Spark signifie seulement « aucun fichier à ce chemin »."""
    return "PATH_NOT_FOUND" in str(erreur)

# 1. Couples déjà présents en landing topscorers (colonnes _param_ = paramètres exacts de l'appel)
deja_charges = set()
try:
    deja = (
        spark.read.parquet(f"{CHEMIN_TABLE}/*/*/*/*.parquet")
        .select("_param_league", "_param_season").distinct()
        .collect()
    )
    deja_charges = {(r["_param_league"], r["_param_season"]) for r in deja}
except Exception as e:
    if not absent(e):
        raise                                                        # autre erreur : on s'arrête
    print(f"ℹ️ Aucun fichier en landing pour {TABLE} (1er lancement ou dossier vidé)")

# 2. Couples terminés, d'après le dernier dépôt fixtures de chaque couple
saisons_terminees = set()
try:
    par_couple = Window.partitionBy("league__id", "league__season")
    terminees = (
        spark.read.parquet(f"{CHEMIN_FIXTURES}/*/*/*/*.parquet")
        .withColumn("_dernier_depot", F.max("_dlt_load_id").over(par_couple))   # dernier dépôt du couple
        .where(F.col("_dlt_load_id") == F.col("_dernier_depot"))                # on ne garde que ce dépôt
        .groupBy("league__id", "league__season")
        .agg(F.min(F.col("fixture__status__short").isin(STATUTS_FINAUX).cast("int")).alias("tous_finis"))
        .where("tous_finis = 1")
        .collect()
    )
    saisons_terminees = {(r["league__id"], r["league__season"]) for r in terminees}
except Exception as e:
    if not absent(e):
        raise
    print("ℹ️ Aucun fichier fixtures en landing : aucune saison considérée comme terminée")

# 3. On n'exclut que ce qui est à la fois déjà chargé ET terminé
a_exclure = deja_charges & saisons_terminees
APPELS = [a for a in APPELS if (a["league"], a["season"]) not in a_exclure]
print(f"Déjà en landing : {len(deja_charges)} | Terminés : {len(saisons_terminees)} | Appels restants : {len(APPELS)}")
if not APPELS:
    dbutils.notebook.exit("Rien à charger : tous les couples demandés sont déjà en landing et terminés.")

## 6. OPTIONNEL — Contrôle du quota (API-Football `/status`, gratuit)

À supprimer pour une API sans quota journalier.

In [0]:
statut = requests.get(f"{BASE_URL}/status", headers=HEADERS).json()["response"]
restantes = statut["requests"]["limit_day"] - statut["requests"]["current"]
print(f"Plan {statut['subscription']['plan']} | {restantes} requêtes restantes")
if not statut["subscription"]["active"] or restantes < len(APPELS):
    raise RuntimeError(f"Quota insuffisant : {len(APPELS)} nécessaires, {restantes} restantes.")

## 7. La ressource dlthub

Une fonction qui **produit** des lignes (`yield`). Chaque ligne reçoit aussi les paramètres de son appel (colonnes `_param_<nom>`), pour savoir d'où elle vient.

In [0]:
def extraire_lignes(reponse_json):
    # API-Football top* : 1 joueur dans reponse_json["response"], avec une liste "statistics"
    # "statistics" = 1 élément par CLUB du joueur dans la compétition × saison (2 si transfert en cours de saison)
    # -> 1 ligne par joueur × club : dlthub aplatit chaque élément en colonnes statistics__... (pas de table enfant)
    lignes = []
    for ligne in reponse_json["response"]:
        stats = ligne.get("statistics") or []
        if not stats:   # garde-fou : un joueur sans statistiques = réponse anormale
            raise RuntimeError(f"statistics vide (joueur {ligne['player']['id']})")
        for stat in stats:
            lignes.append({**ligne, "statistics": stat})   # même joueur, un club à la fois
    return lignes

def verifier_erreurs(reponse_json, params):
    # À ADAPTER : comment l'API signale une erreur ? (API-Football : champ "errors" non vide)

    if reponse_json.get("errors"):
        raise RuntimeError(f"Erreur API ({params}) : {reponse_json['errors']}")

@dlt.resource(name=TABLE, write_disposition="append", columns=TYPES_IMPOSES)   # types imposés pour les colonnes toujours vides
def ressource(appels: list[dict]):
    for i, params in enumerate(appels):
        if i > 0:
            time.sleep(PAUSE_SECONDES)
        reponse_json = requests.get(f"{BASE_URL}{ENDPOINT}", headers=HEADERS, params=params).json()
        verifier_erreurs(reponse_json, params)
        lignes = extraire_lignes(reponse_json)
        print(f"📥 {params} : {len(lignes)} ligne(s)")
        yield [{**ligne, **{f"_param_{k}": v for k, v in params.items()}} for ligne in lignes]

## 8. Pipeline et chargement

`layout` = convention figée `<table>/<aaaa>/<mm>/<jj>/` (dlthub impose `{table_name}` avant la date).

In [0]:
pipeline = dlt.pipeline(
    pipeline_name=f"{SOURCE}_{TABLE}",
    destination=dlt.destinations.filesystem(
        bucket_url=f"file://{LANDING}",
        layout="{table_name}/{YYYY}/{MM}/{DD}/{load_id}.{file_id}.{ext}",
    ),
    dataset_name=SOURCE,
)
info = pipeline.run(ressource(APPELS), loader_file_format="parquet")
print(info)
info.raise_on_failed_jobs()

## 9. Vérifications : fichiers du jour et contenu

In [0]:
from datetime import datetime, timezone
dossier_jour = f"{LANDING}/{SOURCE}/{TABLE}/{datetime.now(timezone.utc):%Y/%m/%d}"
for f in sorted(os.listdir(dossier_jour)):
    print("📄", f)

df = spark.read.parquet(dossier_jour)
print("Colonnes :", len(df.columns), "| Lignes (dépôts du jour) :", df.count())
display(df.limit(20))